In [2]:
import re
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

print("Setup complete")

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/sandali/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /Users/sandali/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /Users/sandali/nltk_data...


Setup complete


In [3]:
job_description = """
We are looking for a Data Analyst with strong skills in Python, SQL, and Excel.
The ideal candidate has experience with pandas, data visualization using matplotlib
or Power BI, and understanding of machine learning concepts such as regression
and classification. Experience with Docker and cloud platforms like AWS is a plus.
Strong communication skills and the ability to work in an agile team environment
are required.
"""

my_cv_text = """
Data Science undergraduate with experience in Python, SQL, pandas, numpy, and
scikit-learn. Built a real-time streaming pipeline using Apache Kafka and Docker.
Trained a Random Forest model achieving strong predictive performance on
marketing campaign data. Experience with Power BI dashboards and PostgreSQL.
Strong problem solving and team collaboration skills.
"""

print("Job description length:", len(job_description.split()))
print("CV length:", len(my_cv_text.split()))

Job description length: 66
CV length: 49


In [4]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    # Step 1: lowercase everything
    text = text.lower()
    
    # Step 2: remove punctuation and numbers, keep only letters and spaces
    text = re.sub(r'[^a-z\s]', ' ', text)
    
    # Step 3: split into individual words
    words = text.split()
    
    # Step 4: remove stopwords (the, and, is, etc.) and short words
    words = [w for w in words if w not in stop_words and len(w) > 2]
    
    # Step 5: lemmatize (reduce words to their root form)
    words = [lemmatizer.lemmatize(w) for w in words]
    
    return ' '.join(words)

# Test it on both texts
cleaned_jd = clean_text(job_description)
cleaned_cv = clean_text(my_cv_text)

print("=== CLEANED JOB DESCRIPTION ===")
print(cleaned_jd)
print("\n=== CLEANED CV ===")
print(cleaned_cv)

=== CLEANED JOB DESCRIPTION ===
looking data analyst strong skill python sql excel ideal candidate experience panda data visualization using matplotlib power understanding machine learning concept regression classification experience docker cloud platform like aws plus strong communication skill ability work agile team environment required

=== CLEANED CV ===
data science undergraduate experience python sql panda numpy scikit learn built real time streaming pipeline using apache kafka docker trained random forest model achieving strong predictive performance marketing campaign data experience power dashboard postgresql strong problem solving team collaboration skill


In [5]:
test_words = ['managing', 'managed', 'manages', 'running', 'analyzed', 'pipelines']
for w in test_words:
    print(f"{w} → {lemmatizer.lemmatize(w, pos='v')}")

managing → manage
managed → manage
manages → manage
running → run
analyzed → analyze
pipelines → pipelines


In [6]:
# Known multi-word terms that must not get split apart
SKILL_PHRASES = {
    'power bi': 'powerbi',
    'machine learning': 'machinelearning',
    'random forest': 'randomforest',
    'spring boot': 'springboot',
    'spring security': 'springsecurity',
    'real time': 'realtime',
    'data science': 'datascience',
    'data analyst': 'dataanalyst',
    'sql server': 'sqlserver',
}

def protect_phrases(text):
    text = text.lower()
    for phrase, token in SKILL_PHRASES.items():
        text = text.replace(phrase, token)
    return text

def clean_text(text):
    text = protect_phrases(text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words and len(w) > 1]  # relaxed to >1 so "BI" alone could survive if needed
    words = [lemmatizer.lemmatize(w) for w in words]
    return ' '.join(words)

cleaned_jd = clean_text(job_description)
cleaned_cv = clean_text(my_cv_text)

print("=== CLEANED JD (FIXED) ===")
print(cleaned_jd)
print("\n=== CLEANED CV (FIXED) ===")
print(cleaned_cv)

=== CLEANED JD (FIXED) ===
looking dataanalyst strong skill python sql excel ideal candidate experience panda data visualization using matplotlib powerbi understanding machinelearning concept regression classification experience docker cloud platform like aws plus strong communication skill ability work agile team environment required

=== CLEANED CV (FIXED) ===
datascience undergraduate experience python sql panda numpy scikit learn built real time streaming pipeline using apache kafka docker trained randomforest model achieving strong predictive performance marketing campaign data experience powerbi dashboard postgresql strong problem solving team collaboration skill


In [7]:
def protect_phrases(text):
    text = text.lower()
    text = text.replace('-', ' ')  # normalize hyphens to spaces first
    for phrase, token in SKILL_PHRASES.items():
        text = text.replace(phrase, token)
    return text

# Re-run cleaning with the fix
cleaned_jd = clean_text(job_description)
cleaned_cv = clean_text(my_cv_text)

print("=== CLEANED CV (HYPHEN FIXED) ===")
print(cleaned_cv)

=== CLEANED CV (HYPHEN FIXED) ===
datascience undergraduate experience python sql panda numpy scikit learn built realtime streaming pipeline using apache kafka docker trained randomforest model achieving strong predictive performance marketing campaign data experience powerbi dashboard postgresql strong problem solving team collaboration skill


In [8]:
documents = [cleaned_jd, cleaned_cv]

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(documents)

print("Matrix shape:", tfidf_matrix.shape)
print("Number of unique words (vocabulary size):", len(vectorizer.get_feature_names_out()))

Matrix shape: (2, 58)
Number of unique words (vocabulary size): 58


In [9]:
import pandas as pd

feature_names = vectorizer.get_feature_names_out()
jd_scores = tfidf_matrix[0].toarray()[0]
cv_scores = tfidf_matrix[1].toarray()[0]

scores_df = pd.DataFrame({
    'word': feature_names,
    'jd_score': jd_scores,
    'cv_score': cv_scores
})

print("=== TOP 10 WORDS IN JOB DESCRIPTION ===")
print(scores_df.sort_values('jd_score', ascending=False).head(10).to_string(index=False))

print("\n=== TOP 10 WORDS IN CV ===")
print(scores_df.sort_values('cv_score', ascending=False).head(10).to_string(index=False))

=== TOP 10 WORDS IN JOB DESCRIPTION ===
           word  jd_score  cv_score
     experience  0.247248  0.249207
         strong  0.247248  0.249207
          skill  0.247248  0.124603
        ability  0.173749  0.000000
     matplotlib  0.173749  0.000000
          ideal  0.173749  0.000000
           like  0.173749  0.000000
        looking  0.173749  0.000000
machinelearning  0.173749  0.000000
           plus  0.173749  0.000000

=== TOP 10 WORDS IN CV ===
        word  jd_score  cv_score
      strong  0.247248  0.249207
  experience  0.247248  0.249207
       model  0.000000  0.175126
 datascience  0.000000  0.175126
  postgresql  0.000000  0.175126
randomforest  0.000000  0.175126
    pipeline  0.000000  0.175126
 performance  0.000000  0.175126
       numpy  0.000000  0.175126
   achieving  0.000000  0.175126


In [10]:
similarity_score = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
match_percentage = round(similarity_score * 100, 1)

print(f"Raw cosine similarity: {similarity_score:.4f}")
print(f"Match Score: {match_percentage}%")

Raw cosine similarity: 0.2773
Match Score: 27.7%


In [11]:
jd_words = set(cleaned_jd.split())
cv_words = set(cleaned_cv.split())

overlap = jd_words & cv_words
only_in_jd = jd_words - cv_words

print("=== WORDS IN BOTH (matched skills) ===")
print(sorted(overlap))

print(f"\n=== WORDS ONLY IN JD (missing from CV) — {len(only_in_jd)} words ===")
print(sorted(only_in_jd))

=== WORDS IN BOTH (matched skills) ===
['data', 'docker', 'experience', 'panda', 'powerbi', 'python', 'skill', 'sql', 'strong', 'team', 'using']

=== WORDS ONLY IN JD (missing from CV) — 23 words ===
['ability', 'agile', 'aws', 'candidate', 'classification', 'cloud', 'communication', 'concept', 'dataanalyst', 'environment', 'excel', 'ideal', 'like', 'looking', 'machinelearning', 'matplotlib', 'platform', 'plus', 'regression', 'required', 'understanding', 'visualization', 'work']


In [12]:
KNOWN_SKILLS = {
    # Languages
    'python', 'sql', 'java', 'javascript',
    # Data & ML
    'panda', 'numpy', 'scikitlearn', 'machinelearning', 'randomforest',
    'regression', 'classification', 'matplotlib', 'seaborn', 'visualization',
    'datascience', 'dataanalyst', 'eda',
    # BI & Tools
    'powerbi', 'excel', 'tableau', 'dashboard',
    # Data Engineering
    'docker', 'kafka', 'realtime', 'pipeline', 'postgresql', 'mongodb',
    'sqlserver', 'cloud', 'aws', 'azure', 'gcp',
    # Backend/Web
    'springboot', 'springsecurity', 'react', 'nodejs', 'api',
    # Soft skills (kept separate for clarity later)
    'communication', 'collaboration', 'agile', 'leadership',
}

def extract_skills(word_set):
    return word_set & KNOWN_SKILLS

jd_skills = extract_skills(jd_words)
cv_skills = extract_skills(cv_words)

matched_skills = jd_skills & cv_skills
missing_skills = jd_skills - cv_skills

print("=== SKILLS REQUIRED BY JD ===")
print(sorted(jd_skills))

print(f"\n=== SKILLS YOU HAVE THAT MATCH — {len(matched_skills)} ===")
print(sorted(matched_skills))

print(f"\n=== SKILLS MISSING FROM YOUR CV — {len(missing_skills)} ===")
print(sorted(missing_skills))

skill_match_pct = round(len(matched_skills) / len(jd_skills) * 100, 1) if jd_skills else 0
print(f"\nSkill Coverage: {skill_match_pct}%")

=== SKILLS REQUIRED BY JD ===
['agile', 'aws', 'classification', 'cloud', 'communication', 'dataanalyst', 'docker', 'excel', 'machinelearning', 'matplotlib', 'panda', 'powerbi', 'python', 'regression', 'sql', 'visualization']

=== SKILLS YOU HAVE THAT MATCH — 5 ===
['docker', 'panda', 'powerbi', 'python', 'sql']

=== SKILLS MISSING FROM YOUR CV — 11 ===
['agile', 'aws', 'classification', 'cloud', 'communication', 'dataanalyst', 'excel', 'machinelearning', 'matplotlib', 'regression', 'visualization']

Skill Coverage: 31.2%


In [13]:
final_score = round(
    (similarity_score * 0.4 + (len(matched_skills) / len(jd_skills) if jd_skills else 0) * 0.6) * 100,
    1
)

print(f"TF-IDF Similarity Score: {match_percentage}%")
print(f"Skill Coverage Score: {skill_match_pct}%")
print(f"Final Weighted Match Score: {final_score}%")

TF-IDF Similarity Score: 27.7%
Skill Coverage Score: 31.2%
Final Weighted Match Score: 29.8%


In [14]:
real_cv_text = """
Data Science undergraduate at SLIIT with hands-on experience across the full data stack,
from real-time streaming pipelines and machine learning to enterprise BI reporting and
full-stack web development. Built an end-to-end real-time data pipeline simulating IoT
sensors, streaming events through a 3-partition Kafka topic into PostgreSQL. Engineered
marketing KPIs from 300,000 raw ad campaign records. Trained a Random Forest classifier
achieving 0.73 ROC-AUC to predict campaign engagement, identifying CPC as the strongest
predictor. Built a Power BI dashboard surfacing price trends. Designed a star schema data
warehouse in PostgreSQL with SQLAlchemy-managed ingestion. Python, pandas, numpy, scikit-learn,
SQL Server, SSIS, SSAS, Power BI, Dimensional Modelling, Star Schema, OLAP, Apache Kafka,
Docker, PostgreSQL, SQLAlchemy, Window Functions, CTEs, cron Automation.
"""

cleaned_real_cv = clean_text(real_cv_text)
real_cv_words = set(cleaned_real_cv.split())
real_cv_skills = extract_skills(real_cv_words)

real_matched = jd_skills & real_cv_skills
real_missing = jd_skills - real_cv_skills

print("Your real CV matched skills:", sorted(real_matched))
print("Your real CV missing skills:", sorted(real_missing))
print(f"Real skill coverage: {round(len(real_matched)/len(jd_skills)*100, 1)}%")

Your real CV matched skills: ['docker', 'machinelearning', 'panda', 'powerbi', 'python']
Your real CV missing skills: ['agile', 'aws', 'classification', 'cloud', 'communication', 'dataanalyst', 'excel', 'matplotlib', 'regression', 'sql', 'visualization']
Real skill coverage: 31.2%


In [15]:
print("Does 'sql' appear in cleaned real CV text?")
print('sql' in real_cv_words)

print("\nLet's see the actual cleaned text:")
print(cleaned_real_cv)

Does 'sql' appear in cleaned real CV text?
False

Let's see the actual cleaned text:
datascience undergraduate sliit hand experience across full data stack realtime streaming pipeline machinelearning enterprise bi reporting full stack web development built end end realtime data pipeline simulating iot sensor streaming event partition kafka topic postgresql engineered marketing kpis raw ad campaign record trained randomforest classifier achieving roc auc predict campaign engagement identifying cpc strongest predictor built powerbi dashboard surfacing price trend designed star schema data warehouse postgresql sqlalchemy managed ingestion python panda numpy scikit learn sqlserver ssis ssa powerbi dimensional modelling star schema olap apache kafka docker postgresql sqlalchemy window function ctes cron automation


In [16]:
# Add the merged multi-word skill tokens to the whitelist too,
# and make sure component skills aren't lost
KNOWN_SKILLS.update({'sqlserver', 'springboot', 'springsecurity', 'randomforest', 'realtime'})

# Also add a manual rule: if sqlserver is present, sql is implied
def expand_implied_skills(skill_set):
    expanded = set(skill_set)
    if 'sqlserver' in expanded:
        expanded.add('sql')
    if 'randomforest' in expanded:
        expanded.add('machinelearning')
    return expanded

real_cv_skills_expanded = expand_implied_skills(extract_skills(real_cv_words))
real_matched_fixed = jd_skills & real_cv_skills_expanded
real_missing_fixed = jd_skills - real_cv_skills_expanded

print("Fixed matched skills:", sorted(real_matched_fixed))
print("Fixed missing skills:", sorted(real_missing_fixed))
print(f"Fixed skill coverage: {round(len(real_matched_fixed)/len(jd_skills)*100, 1)}%")

Fixed matched skills: ['docker', 'machinelearning', 'panda', 'powerbi', 'python', 'sql']
Fixed missing skills: ['agile', 'aws', 'classification', 'cloud', 'communication', 'dataanalyst', 'excel', 'matplotlib', 'regression', 'visualization']
Fixed skill coverage: 37.5%
